# Phase 1 — Notebook 1.5: Semantic Enrichment

Builds two injection maps from the preprocessed vocab, then writes an enriched parquet
that `02_tfidf_analysis.ipynb` can load in place of `05_filtered.parquet`.

| Pass | What | Output |
|------|------|--------|
| A | Subject clustering (SentenceTransformer + agglomerative) | `semantic_map.csv` |
| B | Framing taxonomy discovery + classification (LLM) | `framing_map.csv` |
| C | Token injection | `05_enriched.parquet` |

**Run order:** `01_preprocess` → `01.5_semantic_enrichment` → `02_tfidf_analysis`

**Key outputs (all human-reviewable before injection runs):**
- `OUTPUTS/enrichment/semantic_map.csv` — term → primary_category, subcategory
- `OUTPUTS/enrichment/framing_taxonomy.json` — discovered framing categories (edit before classifying)
- `OUTPUTS/enrichment/framing_map.csv` — term → framing_category
- `OUTPUTS/enrichment/injection_preview.csv` — full preview of what gets added to each unique token set

## Setup

In [1]:
import sys, os, json, time, hashlib, pickle, unicodedata, re
from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import (load_cfg, flat_freq, build_output_path,
                   get_llm_client, slugify_group_value)

CFG = load_cfg()

def OUT(subdir, fname):
    return build_output_path(subdir, fname)  # flat — enrichment outputs are corpus-level

client          = get_llm_client()
MODEL_DISCOVERY = CFG["models"]["discovery"]
MODEL_CLASSIFY  = CFG["models"]["classify"]
MODEL_GATE      = CFG["models"]["gate"]

df       = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_filtered.parquet")
doc_freq = pd.read_csv(ROOT / "OUTPUTS/vocab/unigram_docfreq.csv",
                       index_col=0).squeeze().rename("doc_freq")
freq     = flat_freq(df).rename("corpus_freq")
vocab_df = pd.DataFrame({"corpus_freq": freq, "doc_freq": doc_freq}).dropna()
vocab_df["docs_pct"] = vocab_df["doc_freq"] / len(df)

print(f"Corpus:        {len(df):,} projects")
print(f"Full vocab:    {len(vocab_df):,} terms")
print(f"Rare (<1k df): {(vocab_df.doc_freq < 1000).sum():,} terms")


Corpus:        896,277 projects
Full vocab:    7,297 terms
Rare (<1k df): 3,622 terms


---
## Pass A — Subject clustering

Embeds rare vocab terms with SentenceTransformer, clusters with agglomerative cosine distance,
then runs each cluster through an LLM coherence gate. Only `inject` clusters become tokens.

In [2]:
RARE_DOC_FREQ_CEILING  = CFG["enrichment"]["rare_doc_freq_ceiling"]
AGG_DISTANCE_THRESHOLD = CFG["enrichment"]["agg_distance_threshold"]
MIN_CLUSTER_SIZE       = CFG["enrichment"]["min_cluster_size"]
EMBEDDING_MODEL        = CFG["enrichment"]["embedding_model"]
CACHE_SCHEMA_VERSION   = CFG["enrichment"]["cache_schema_version"]

rare_terms = vocab_df[vocab_df.doc_freq < RARE_DOC_FREQ_CEILING].index.tolist()
print(f"Rare terms to cluster: {len(rare_terms):,}")

# Cache key includes: term set, model, clustering params, schema version.
cache_components = "|".join([
    hashlib.md5(",".join(sorted(rare_terms)).encode()).hexdigest(),
    EMBEDDING_MODEL,
    str(AGG_DISTANCE_THRESHOLD),
    str(MIN_CLUSTER_SIZE),
    CACHE_SCHEMA_VERSION,
])
cache_key  = hashlib.md5(cache_components.encode()).hexdigest()
CACHE_DIR  = ROOT / ".cache"
CACHE_DIR.mkdir(exist_ok=True)
cache_path = CACHE_DIR / f"embeddings_clusters_{cache_key[:12]}.pkl"

import httpx
import urllib3

os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

if not getattr(httpx.Client, "_dc_verify_patched", False):
    _original_init = httpx.Client.__init__

    def _patched_init(self, *args, **kwargs):
        kwargs["verify"] = False
        return _original_init(self, *args, **kwargs)

    httpx.Client.__init__ = _patched_init
    httpx.Client._dc_verify_patched = True

urllib3.disable_warnings()

if cache_path.exists():
    print(f"Cache hit ({cache_key[:8]}) — loading embeddings and cluster labels...")
    with open(cache_path, "rb") as f:
        embeddings, labels = pickle.load(f)
else:
    print(f"Cache miss — embedding {len(rare_terms):,} terms with {EMBEDDING_MODEL}...")
    embedder   = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = embedder.encode(rare_terms, show_progress_bar=True,
                                 normalize_embeddings=True)
    agg = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=AGG_DISTANCE_THRESHOLD,
        metric="cosine",
        linkage="average",
    )
    labels = agg.fit_predict(embeddings)
    with open(cache_path, "wb") as f:
        pickle.dump((embeddings, labels), f)
    print(f"Written to cache: {cache_path.name}")

sizes      = Counter(labels)
valid_ids  = {cid for cid, n in sizes.items() if n >= MIN_CLUSTER_SIZE}
cluster_df = pd.DataFrame({"term": rare_terms, "cluster_id": labels})
cluster_df = cluster_df[cluster_df.cluster_id.isin(valid_ids)].copy()

clusters = (cluster_df.groupby("cluster_id")["term"]
            .apply(list).reset_index(name="terms"))

print(f"\nTotal terms in valid clusters: {len(cluster_df):,} of {len(rare_terms):,}")
print(f"Valid clusters ({MIN_CLUSTER_SIZE}+ members): {len(clusters)}")
print("\nSample clusters:")
for _, row in clusters.sample(min(8, len(clusters)), random_state=42).iterrows():
    print(f"  C{row.cluster_id:03d}: {row.terms[:12]}")


Rare terms to cluster: 3,622
Cache hit (9b301ba2) — loading embeddings and cluster labels...

Total terms in valid clusters: 294 of 3,622
Valid clusters (3+ members): 90

Sample clusters:
  C148: ['freeze', 'freezer', 'freezing']
  C075: ['corporate', 'corporation', 'incorporation']
  C220: ['gene', 'genetic', 'genetics']
  C283: ['el', 'elapse', 'elar', 'elate', 'eld']
  C004: ['guatemala', 'honduras', 'venezuela']
  C081: ['bilingualism', 'lingual', 'monolingual']
  C140: ['battleship', 'naval', 'navy']
  C267: ['dodge', 'dodgeball', 'dodgeballs']


In [3]:
# ── LLM coherence gate ────────────────────────────────────────────────────────
# One call per cluster. Returns inject / split / discard + proposed labels.

GATE_SYSTEM = (
    "You are an NLP analyst reviewing term clusters from DonorsChoose teacher project "
    "essays (K-12 US public school classrooms). These clusters come from rare vocabulary "
    "(terms appearing in fewer than 1,000 of 900,000 projects). "
    "Respond ONLY with valid JSON, no preamble, no markdown fences."
)

GATE_PROMPT = """Cluster {cid} terms: {terms}

Assess whether these terms share a coherent educational domain concept.

Return a JSON object with exactly these keys:
  action           : "inject" | "split" | "discard"
  primary_category : snake_case domain label (e.g. "marine_biology") or null
  subcategory      : more specific snake_case label or null
  split_into       : list of {{"label": ..., "terms": [...]}} if action=split, else []
  reasoning        : one sentence

action=inject  : terms clearly share a specific educational domain concept worth surfacing
action=split   : 2-3 coherent sub-groups exist within this cluster
action=discard : generic language, morphological noise, or no meaningful shared concept

Be conservative — discard ambiguous clusters. A useful injection token must be specific
enough that its presence changes what a TF-IDF topic means."""


def gate_cluster(cid, terms, retries=2):
    prompt = GATE_PROMPT.format(cid=cid, terms=terms[:20])
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_GATE,
                messages=[
                    {"role": "system", "content": GATE_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
            )
            text = resp.choices[0].message.content.strip()
            return json.loads(text)
        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                return {"action": "discard", "primary_category": None,
                        "subcategory": None, "split_into": [],
                        "reasoning": f"API error: {e}"}


gate_results = []
for i, row in clusters.iterrows():
    result = gate_cluster(row.cluster_id, row.terms)
    result["cluster_id"] = row.cluster_id
    result["terms"]      = row.terms
    gate_results.append(result)
    action = result.get("action", "?")
    label  = result.get("primary_category", "")
    if (i % 10 == 0) or action != "discard":
        print(f"  C{row.cluster_id:03d} [{action:7s}] {label or result.get('reasoning','')[:60]}")

gate_df = pd.DataFrame(gate_results)
print(f"\nGate summary:")
print(gate_df["action"].value_counts().to_string())

  C004 [discard] The terms represent names of countries and lack a coherent e
  C013 [inject ] global_education
  C014 [inject ] marine_navigation
  C025 [inject ] educational_infrastructure
  C035 [inject ] history
  C036 [inject ] media_studies
  C042 [inject ] criminal_justice
  C066 [inject ] education_assessment
  C067 [discard] The terms 'epidemic', 'outbreak', and 'plague' are too broad
  C069 [inject ] literary_analysis
  C081 [inject ] linguistics
  C086 [inject ] language_arts
  C094 [inject ] music_education
  C100 [discard] The terms are too generic and do not represent a coherent ed
  C129 [inject ] physical_education
  C132 [inject ] linguistics
  C133 [inject ] education
  C148 [discard] The terms do not represent a coherent educational domain con
  C151 [inject ] physical_education
  C153 [inject ] microbiology
  C160 [inject ] geography
  C199 [inject ] adolescent_development
  C208 [inject ] geography
  C220 [inject ] biology
  C237 [inject ] literature
  C253 [discar

In [4]:
# ── Build semantic_map.csv ────────────────────────────────────────────────────
# Expand inject + split results into a flat term → category mapping.

sem_rows = []

for _, row in gate_df.iterrows():
    if row.action == "inject" and row.primary_category:
        for term in row.terms:
            sem_rows.append({
                "term": term,
                "primary_category": row.primary_category,
                "subcategory": row.get("subcategory"),
                "source_cluster": row.cluster_id
            })

    elif row.action == "split" and row.get("split_into"):
        for sub in row["split_into"]:
            for term in sub.get("terms", []):
                sem_rows.append({
                    "term": term,
                    "primary_category": sub["label"],
                    "subcategory": None,
                    "source_cluster": row.cluster_id
                })

semantic_map = pd.DataFrame(sem_rows)
semantic_map.to_csv(OUT("enrichment", "semantic_map.csv"), index=False)

print(f"semantic_map.csv: {len(semantic_map):,} terms mapped")
print(f"Unique primary categories: {semantic_map.primary_category.nunique()}")
print("\nTop categories by term count:")
print(semantic_map.primary_category.value_counts().head(20).to_string())

semantic_map.csv: 106 terms mapped
Unique primary categories: 31

Top categories by term count:
primary_category
linguistics                6
geography                  6
physical_education         6
criminal_justice           4
asian_studies              4
global_education           4
aerospace_engineering      4
psychology                 3
audio_engineering          3
physical_science           3
technology_in_education    3
education_resources        3
art_education              3
ecology                    3
ethics_in_education        3
criminal_investigation     3
art_appreciation           3
adolescent_development     3
literature                 3
biology                    3


---
## Pass B — Framing taxonomy discovery

Two steps:
1. Send a representative vocab sample to the LLM — it proposes a taxonomy of framing/tone/context categories specific to DonorsChoose essay language
2. Classify the full vocab against that taxonomy in batches

**You review and edit `framing_taxonomy.json` between steps 1 and 2.**

In [5]:
# ── Step B1: Load curated taxonomy ───────────────────────────────────────────
# The framing taxonomy has been developed through an external research and review
# process and is loaded directly from file. The original LLM-based discovery call
# that previously lived here has been removed so it cannot overwrite the curated
# taxonomy. To update the taxonomy, edit the JSON file directly and re-run from
# this cell onward.
#
# Expected location: OUTPUTS/enrichment/framing_taxonomy.json
# Copy your curated file there before running.

taxonomy_path = OUT("enrichment", "framing_taxonomy.json")
if not taxonomy_path.exists():
    raise FileNotFoundError(
        f"framing_taxonomy.json not found at {taxonomy_path}\n"
        "Copy your curated taxonomy file to OUTPUTS/enrichment/ before running."
    )

with open(taxonomy_path) as f:
    taxonomy = json.load(f)

nmf_cats      = [c for c in taxonomy if c.get('tier') == 'nmf']
analysis_cats = [c for c in taxonomy if c.get('tier') == 'analysis']

print(f"Loaded {len(taxonomy)} categories from {taxonomy_path}")
print(f"  NMF-tier:      {len(nmf_cats)}")
print(f"  Analysis-tier: {len(analysis_cats)}")
print()
for cat in taxonomy:
    tier = cat.get('tier', '?')
    cov  = cat.get('target_coverage_pct', '?')
    print(f"  [{tier:8s}] {cat['category']:50s} {cov}")


Loaded 63 categories from /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/enrichment/framing_taxonomy.json
  NMF-tier:      15
  Analysis-tier: 48

  [nmf     ] episodic_disruption_recovery                       20-35%
  [nmf     ] chronic_scarcity                                   18-32%
  [nmf     ] barrier_removal                                    25-40%
  [nmf     ] catch_up_remediation                               18-30%
  [nmf     ] opportunity_expansion                              20-35%
  [nmf     ] capacity_first_dignity                             18-30%
  [nmf     ] regulation_safety_need                             18-32%
  [nmf     ] readiness_to_learn                                 20-35%
  [nmf     ] loss_avoidance                                     18-32%
  [nmf     ] urgent_now_anchoring                               15-25%
  [nmf     ] future_preparation                                 18-30%
  [nmf     ] household_basic_needs_spillover       

In [6]:
# ── Step B2: Classify full vocab in batches ───────────────────────────────────
# Classify all vocab terms (not just rare — framing appears in head vocab too).
# Most terms return null. ~5-10% expected to match a category.

BATCH_SIZE = CFG["enrichment"]["classify_batch_size"]
FORCE_RECLASSIFY  = CFG["enrichment"]["force_reclassify"]
framing_map_path  = OUT("enrichment", "framing_map.csv")
framing_meta_path = OUT("enrichment", "framing_map_meta.json")

# Hash the taxonomy that drove (or will drive) this classification run
taxonomy_source = OUT("enrichment", "framing_taxonomy.json")
with open(taxonomy_source) as _f:
    current_taxonomy_hash = hashlib.md5(_f.read().encode()).hexdigest()

def _meta_hash_matches():
    if not framing_meta_path.exists():
        return False
    with open(framing_meta_path) as _mf:
        meta = json.load(_mf)
    return meta.get("taxonomy_md5") == current_taxonomy_hash

# Build taxonomy reference string for the prompt
def _fmt_cat(cat):
    lines = [
        f"  [{cat.get('tier','?')}] {cat['category']} (target: {cat.get('target_coverage_pct','?')})",
        f"    Description: {cat['description']}",
        f"    Examples: {', '.join(cat.get('examples', [])[:8])}",
        f"    Counter-examples (must return null): {', '.join(cat.get('counter_examples', [])[:6])}",
    ]
    return '\n'.join(lines)

taxonomy_ref = "\n\n".join(_fmt_cat(cat) for cat in taxonomy)
valid_categories = {cat["category"] for cat in taxonomy}

CLASSIFY_SYSTEM = (
    "You classify vocabulary terms from DonorsChoose teacher essays into framing/tone categories. "
    "Respond ONLY with a JSON object mapping term to category (or null). No preamble."
)

CLASSIFY_PROMPT = """Framing categories:
{taxonomy}

Classify each term below. Return null if the term doesn't clearly fit a framing/tone category
(most subject matter words, nouns, and neutral verbs should be null).
Only classify when the framing signal is clear and consistent across different essay contexts.

Terms: {terms}

Return only: {{"term1": "category_or_null", "term2": "category_or_null", ...}}"""

def classify_batch(terms_batch, retries=2):
    prompt = CLASSIFY_PROMPT.format(
        taxonomy=taxonomy_ref,
        terms=terms_batch
    )
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_CLASSIFY,
                messages=[
                    {"role": "system", "content": CLASSIFY_SYSTEM},
                    {"role": "user",   "content": prompt},
                ],
            )
            text = resp.choices[0].message.content.strip()
            raw  = json.loads(text)
            return {
                k: v for k, v in raw.items()
                if v is None or v in valid_categories
            }
        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                print(f"    Batch error: {e}")
                return {t: None for t in terms_batch}

if framing_map_path.exists() and _meta_hash_matches() and not FORCE_RECLASSIFY:
    framing_map = pd.read_csv(framing_map_path)
    print(f"Reusing existing framing_map.csv ({len(framing_map):,} terms) — "
          f"taxonomy hash matches. Set force_reclassify: true to override.")
elif framing_map_path.exists() and not _meta_hash_matches() and not FORCE_RECLASSIFY:
    print("WARNING: framing_map.csv exists but taxonomy hash has changed.")
    print("         Re-running classification. Set force_reclassify: false to suppress.")
    all_terms   = vocab_df.index.tolist()
    batches     = [all_terms[i:i+BATCH_SIZE] for i in range(0, len(all_terms), BATCH_SIZE)]
    framing_raw = {}

    print(f"Classifying {len(all_terms):,} terms in {len(batches)} batches...")
    for i, batch in enumerate(batches):
        result = classify_batch(batch)
        framing_raw.update(result)
        if (i + 1) % 10 == 0:
            classified = sum(1 for v in framing_raw.values() if v is not None)
            print(f"  Batch {i+1}/{len(batches)} — {classified:,} terms classified so far")

    framing_map = pd.DataFrame([
        {"term": term, "framing_category": cat}
        for term, cat in framing_raw.items() if cat is not None
    ])
    framing_map.to_csv(framing_map_path, index=False)
else:
    all_terms   = vocab_df.index.tolist()
    batches     = [all_terms[i:i+BATCH_SIZE] for i in range(0, len(all_terms), BATCH_SIZE)]
    framing_raw = {}

    print(f"Classifying {len(all_terms):,} terms in {len(batches)} batches...")
    for i, batch in enumerate(batches):
        result = classify_batch(batch)
        framing_raw.update(result)
        if (i + 1) % 10 == 0:
            classified = sum(1 for v in framing_raw.values() if v is not None)
            print(f"  Batch {i+1}/{len(batches)} — {classified:,} terms classified so far")

    framing_map = pd.DataFrame([
        {"term": term, "framing_category": cat}
        for term, cat in framing_raw.items() if cat is not None
    ])
    framing_map.to_csv(framing_map_path, index=False)

# Write provenance meta file
meta = {
    "taxonomy_md5":       current_taxonomy_hash,
    "model":              MODEL_CLASSIFY,
    "classified_at":      datetime.now().isoformat(),
    "n_terms_classified": int(len(framing_map)),
}
with open(framing_meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nframing_map.csv: {len(framing_map):,} terms classified")
print(f"Coverage: {len(framing_map)/len(vocab_df):.1%} of vocab")
print("\nCategory distribution:")
print(framing_map.framing_category.value_counts().to_string())

# Coverage check against documented targets
warn_threshold = CFG["enrichment"]["coverage_deviation_warn_threshold"]
actual_pct_by_cat = framing_map["framing_category"].value_counts(normalize=True).mul(100).to_dict()

def _target_midpoint(v):
    if v is None:
        return None
    s = str(v).strip().replace("%", "")
    if "-" in s:
        lo, hi = s.split("-", 1)
        try:
            return (float(lo) + float(hi)) / 2
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

for cat in taxonomy:
    target = _target_midpoint(cat.get("target_coverage_pct"))
    if target is None:
        continue
    actual = actual_pct_by_cat.get(cat["category"], 0.0)
    if abs(actual - target) > warn_threshold:
        print(
            f"WARNING: coverage deviation for {cat['category']}: "
            f"actual={actual:.1f}% vs target≈{target:.1f}%"
        )


Classifying 7,297 terms in 49 batches...
  Batch 10/49 — 22 terms classified so far
  Batch 20/49 — 34 terms classified so far
  Batch 30/49 — 86 terms classified so far
  Batch 40/49 — 131 terms classified so far
    Batch error: Expecting value: line 1 column 1 (char 0)

framing_map.csv: 149 terms classified
Coverage: 2.0% of vocab

Category distribution:
framing_category
capacity_first_dignity                15
household_basic_needs_spillover       10
event_news_disaster_anchoring          9
safety_protection_language             8
underfunding_inequity_context          6
collaborative_social_learning          6
barrier_removal                        5
empathy_invitation                     5
catch_up_remediation                   5
joy_possibility_appeal                 4
episodic_disruption_recovery           4
consumables_replenishment              4
loss_avoidance                         4
triage_repair                          3
representation_cultural_reflection     3
chronic_

---
## Pass C — Token injection

Adds `__cat__`, `__sub__`, and `__framing__` tokens to each project's token list.
Original tokens are preserved — injection is additive.

Token naming convention:
- Subject: `__cat_marine_biology__`  and  `__sub_ocean_ecosystems__`
- Framing: `__framing_urgency__`

The double-underscore prefix ensures injected tokens are visually distinct in any output
and can be filtered or analyzed separately.

In [7]:
# ── Build injection lookups ───────────────────────────────────────────────────

semantic_map  = pd.read_csv(OUT("enrichment", "semantic_map.csv"))
framing_map   = pd.read_csv(OUT("enrichment", "framing_map.csv"))

MAX_DOC_FREQ_FOR_INJECTION = CFG["enrichment"]["max_doc_freq_for_injection"]

invalid_cats = set(framing_map.framing_category) - valid_categories
if invalid_cats:
    raise ValueError(
        f"framing_map.csv contains categories not present in framing_taxonomy.json: "
        f"{sorted(invalid_cats)}"
    )

# Terms that bypass the ceiling because they are definitionally load-bearing
# for a specific category. Format: term -> framing_category_name.
# A term listed here injects its paired category even if doc_freq exceeds
# MAX_DOC_FREQ_FOR_INJECTION. Only applies to framing tokens, not cat/sub tokens.
CEILING_EXCEPTIONS = {

    # barrier_removal — 'access' is the single most central term in this category
    'access':          'barrier_removal',
    'block':           'barrier_removal',

    # catch_up_remediation — comprehension anchors literacy remediation framing
    'comprehension':   'catch_up_remediation',

    # chronic_scarcity — lack and low are the core persistent-deprivation vocabulary
    'lack':            'chronic_scarcity',
    'low':             'chronic_scarcity',

    # collaborative_social_learning — collaborate is definitionally central
    'collaborate':     'collaborative_social_learning',
    'collaboration':   'collaborative_social_learning',

    # durable_infrastructure — chairs and boards are the quintessential durable items
    'board':           'durable_infrastructure',
    'chair':           'durable_infrastructure',

    # empathy_invitation — heart is the emotional anchor
    'heart':           'empathy_invitation',

    # event_news_disaster_anchoring — covid and pandemic ARE this category
    'covid':           'event_news_disaster_anchoring',
    'pandemic':        'event_news_disaster_anchoring',
    'event':           'event_news_disaster_anchoring',

    # future_preparation — forward-orientation anchors
    'advance':         'future_preparation',
    'beyond':          'future_preparation',

    # gratitude_orientation — 'thank' is the anchor term; appreciate and gift follow
    'thank':           'gratitude_orientation',
    'appreciate':      'gratitude_orientation',
    'gift':            'gratitude_orientation',

    # home_extension_family_colearning — family is the only high-freq signal
    'family':          'home_extension_family_colearning',

    # humble_plea — hope and please are load-bearing plea language
    'hope':            'humble_plea',
    'please':          'humble_plea',

    # inquiry_exploration — curiosity vocabulary is definitional
    'curious':         'inquiry_exploration',
    'curiosity':       'inquiry_exploration',
    'discover':        'inquiry_exploration',

    # mandate_compliance — curriculum is the core compliance anchor
    'curriculum':      'mandate_compliance',

    # opportunity_expansion — expand and engineering are the key enrichment signals
    'expand':          'opportunity_expansion',
    'engineering':     'opportunity_expansion',

    # regulation_safety_need — calm is the single most central term
    'calm':            'regulation_safety_need',

    # representation_cultural_reflection — culture is definitional
    'culture':         'representation_cultural_reflection',

    # rural_scarcity_dispersion — rural is literally definitional
    'rural':           'rural_scarcity_dispersion',

    # student_choice_autonomy — all four are core choice vocabulary
    'choice':          'student_choice_autonomy',
    'choose':          'student_choice_autonomy',
    'independent':     'student_choice_autonomy',
    'independently':   'student_choice_autonomy',
    'option':          'student_choice_autonomy',

    # system_bundle_request — set and kit are the primary bundle nouns
    'kit':             'system_bundle_request',
    'set':             'system_bundle_request',

    # teacher_as_caregiver — care and encourage are load-bearing
    'care':            'teacher_as_caregiver',
    'encourage':       'teacher_as_caregiver',

    # underfunding_inequity_context — poverty and funding are load-bearing
    'poverty':         'underfunding_inequity_context',
    'funding':         'underfunding_inequity_context',
    'socioeconomic':   'underfunding_inequity_context',

    # urgent_now_anchoring — core urgency vocabulary
    'critical':        'urgent_now_anchoring',
    'now':             'urgent_now_anchoring',
}

inject_lookup = {}

for _, row in semantic_map.iterrows():
    if vocab_df.loc[row.term, 'doc_freq'] > MAX_DOC_FREQ_FOR_INJECTION:
        continue
    tokens = [f"__cat_{row.primary_category}__"]
    if pd.notna(row.get("subcategory")) and row.subcategory:
        tokens.append(f"__sub_{row.subcategory}__")
    inject_lookup.setdefault(row.term, []).extend(tokens)

for _, row in framing_map.iterrows():
    term = row.term
    cat  = row.framing_category
    if vocab_df.loc[term, 'doc_freq'] > MAX_DOC_FREQ_FOR_INJECTION:
        if CEILING_EXCEPTIONS.get(term) != cat:
            continue
    inject_lookup.setdefault(term, []).append(f"__framing_{cat}__")

inject_lookup = {t: list(dict.fromkeys(toks))
                 for t, toks in inject_lookup.items()}


In [8]:
# ── Estimate injected token doc_freq before writing ───────────────────────────
# Any injected token that won't survive min_doc_count filter is flagged here.

min_doc_count = CFG["sql"]["min_doc_count"]

# Count how many projects contain at least one term mapping to each injected token
injected_token_counts = {}
for tokens_list in df["tokens"]:
    seen = set()
    for t in tokens_list:
        if t in inject_lookup:
            for inj in inject_lookup[t]:
                seen.add(inj)
    for inj in seen:
        injected_token_counts[inj] = injected_token_counts.get(inj, 0) + 1

inj_freq_df = pd.DataFrame([
    {"injected_token": tok, "project_count": cnt}
    for tok, cnt in injected_token_counts.items()
]).sort_values("project_count", ascending=False)

will_survive = inj_freq_df[inj_freq_df.project_count >= min_doc_count]
will_drop    = inj_freq_df[inj_freq_df.project_count < min_doc_count]

print(f"Injected tokens surviving min_doc_count={min_doc_count}: {len(will_survive)}")
if len(will_drop):
    print(f"WARNING: {len(will_drop)} injected tokens below threshold (will be filtered):")
    print(f"  {will_drop.injected_token.tolist()}")
print("\nTop injected tokens by project coverage:")
print(will_survive.head(20).to_string(index=False))


Injected tokens surviving min_doc_count=15: 92

Top injected tokens by project coverage:
                             injected_token  project_count
          __framing_system_bundle_request__          98739
         __framing_capacity_first_dignity__          80889
  __framing_collaborative_social_learning__          77630
__framing_household_basic_needs_spillover__          66052
           __framing_teacher_as_caregiver__          65263
                    __framing_humble_plea__          64118
        __framing_student_choice_autonomy__          52277
      __framing_rural_scarcity_dispersion__          45559
            __framing_inquiry_exploration__          38659
  __framing_event_news_disaster_anchoring__          36643
                __framing_barrier_removal__          34917
         __framing_regulation_safety_need__          33849
          __framing_opportunity_expansion__          29198
     __framing_safety_protection_language__          27731
         __framing_joy_pos

In [9]:
# ── Write enriched parquet ────────────────────────────────────────────────────

def inject_tokens(token_list, lookup):
    extra = []
    for t in token_list:
        if t in lookup:
            extra.extend(lookup[t])
    if not extra:
        return token_list
    # Deduplicate injected tokens (original tokens may repeat — that's fine)
    extra_dedup = list(dict.fromkeys(extra))
    return token_list + extra_dedup


df_enriched = df.copy()
# Convert to plain Python lists first — parquet round-trips can produce
# numpy arrays which cause shape-mismatch errors in inject_tokens.
df_enriched["tokens"] = df_enriched["tokens"].apply(
    lambda ts: inject_tokens(list(ts), inject_lookup)
)

# Stats
orig_len = df["tokens"].apply(len)
new_len  = df_enriched["tokens"].apply(len)
enriched_projects = (new_len > orig_len).sum()

print(f"Projects with at least one injected token: {enriched_projects:,} "
      f"({enriched_projects/len(df):.1%})")
print(f"Mean tokens before injection: {orig_len.mean():.1f}")
print(f"Mean tokens after injection:  {new_len.mean():.1f}")
print(f"Max injected tokens on any project: {(new_len - orig_len).max()}")

out_path = ROOT / "OUTPUTS/prepared/05_enriched.parquet"
df_enriched.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}")
print("\nTo use in 02_tfidf_analysis.ipynb, change the load line to:")
print('  df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")')

Projects with at least one injected token: 605,060 (67.5%)
Mean tokens before injection: 58.5
Mean tokens after injection:  59.6
Max injected tokens on any project: 9

Saved → OUTPUTS/prepared/05_enriched.parquet

To use in 02_tfidf_analysis.ipynb, change the load line to:
  df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")
